# RMSNorm

源码导航：[`core/norm/rmsnorm.py`](../../../core/norm/rmsnorm.py) 中的 `RMSNorm`。

LayerNorm 通过减去均值并除以标准差将每层的激活白化，但已有研究表明中心化操作对训练稳定性的贡献远小于尺度归一化。Zhang & Sennrich (2019) 在此基础上提出 **RMSNorm**：仅保留均方根缩放，去掉中心化操作与 bias 参数。在 LLaMA / T5 等 1B 量级以上的 decoder-only 模型中，RMSNorm 已作为 Pre-norm 的默认选择，Walkie 同样遵循此做法。

### 1. 理论推导

设 $x \in \mathbb{R}^d$ 为某一 token 的隐状态向量，其均方根（RMS）定义为：

$$
\operatorname{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2 + \epsilon}
$$

其中 $\epsilon$ 为防止除零的数值稳定项（Walkie 中默认 $10^{-6}$）。RMSNorm 的输出为：

$$
\operatorname{RMSNorm}(x) = \frac{x}{\operatorname{RMS}(x)} \odot w
$$

其中 $w \in \mathbb{R}^d$ 是可学习的缩放参数，初始化为全 $\mathbf{1}$。

**与 LayerNorm 的对比：**

| 属性 | LayerNorm | RMSNorm |
|---|---|---|
| 中心化（减均值） | ✓ | ✗ |
| 可学习 scale $w$ | ✓ | ✓ |
| 可学习 bias $b$ | ✓（可选）| ✗ |
| 每维参数量（dim=$d$）| $2d$（含 bias）| $d$ |

### 2. Walkie 的 fp32 上溯实现细节

在 bf16/fp16 混合精度训练中，直接对低精度张量求平方再求均值容易引入数值误差。Walkie 的实现将输入先提升到 `float32` 计算 RMS，再将结果投影回原 dtype，避免在低精度域的数值溢出或精度丢失。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.rmsnorm import RMSNorm

### 3. 形状与 RMS 数值检查

In [ ]:
torch.manual_seed(0)
# 构造均值不为零、方差约为 3 的随机激活模拟训练中间层的输出
x = torch.randn(2, 4, 16) * 3
norm = RMSNorm(16, eps=1e-6)
y = norm(x)

print("x.shape =", tuple(x.shape))
print("y.shape =", tuple(y.shape))
# 归一化后，每 token 的 RMS 应接近 1（weight 初始化为 1）
print("RMS before:", x.pow(2).mean(dim=-1).sqrt().mean().item())
print("RMS after :", y.pow(2).mean(dim=-1).sqrt().mean().item())
assert x.shape == y.shape, "RMSNorm 必须保持输入输出维度一致！"

### 4. 与 LayerNorm 的参数量对比

In [ ]:
import torch.nn as nn

# 以 Walkie 实际隐维度 1536 为例
dim = 1536
rms = RMSNorm(dim)
ln  = nn.LayerNorm(dim, elementwise_affine=True)

rms_params = sum(p.numel() for p in rms.parameters())
ln_params  = sum(p.numel() for p in ln.parameters())

print(f"RMSNorm params:  {rms_params:,}  (仅 scale w)")
print(f"LayerNorm params:{ln_params:,}  (scale w + bias b)")
print(f"参数节省率: {(1 - rms_params / ln_params) * 100:.0f}%")

### 5. 源码精讲

以下为 `core/norm/rmsnorm.py` 中 `RMSNorm` 的完整实现，并加注与公式的对应关系：

```python
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-6) -> None:
        super().__init__()
        # 可学习缩放向量 w，对应公式中的 w ∈ R^d，初始化为全 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.eps = eps                           # ε，数值稳定项
        self.normalized_shape = normalized_shape # d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 混合精度兼容：先转 fp32 计算 RMS，避免 bf16/fp16 下溢
        orig_dtype = x.dtype
        x_fp32 = x.float()

        # rsqrt 等价于 1/sqrt(·)，计算 1/RMS(x)
        # pow(2).mean(dim=-1, keepdim=True) 对最后一维（特征维 d）求均方
        rms = x_fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()

        # 归一化后投影回原 dtype，乘以可学习 scale w
        out = (x_fp32 * rms).to(orig_dtype)
        return out * self.weight.to(orig_dtype)
```

关键设计点：
- `keepdim=True` 使 `rms` 形状为 `(B, T, 1)`，可直接与 `x_fp32` `(B, T, d)` 广播相乘。
- `weight` 仅有 $d$ 个参数，无 bias，与理论推导一致。
- 对每个 token（最后一维）独立归一化，不涉及跨 token 的均值计算。

---

## 延伸阅读与参考资料

### 核心论文
- **Root Mean Square Layer Normalization**: Zhang and Sennrich, 2019. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467)
- **T5**: Raffel et al., 2019. [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)

### 工程实现
- **Hugging Face Transformers LlamaRMSNorm**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- **PyTorch LayerNorm API**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)